In [ ]:
# =====================================================================
# Dynamic-Room 3-Way Validation
#   A: Pure NSGA-II (200g physics) — ground truth
#   B: LGBM+NSGA-II (200g surrogate → physics verify)
#   C: LGBM+AGE-MOEA (200g surrogate → physics verify)
#   Metrics: HV dev, IGD, wall clock
# =====================================================================

import torch, numpy as np, math, time, json, glob, os
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
import lightgbm as lgb
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.age import AGEMOEA
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.termination import get_termination
from pymoo.optimize import minimize
from pymoo.indicators.hv import HV
from pymoo.indicators.igd import IGD

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ==========================================
# Load LGBM
# ==========================================
def find_file(p):
    for c in [p,f'/kaggle/working/{p}']+sorted(glob.glob(f'/kaggle/input/**/{p}',recursive=True)):
        if os.path.exists(c):return c
    raise FileNotFoundError(p)
lgbm_m=lgb.Booster(model_file=find_file('lgbm_dynamic.txt'))

def enrich(xc,zc,Lx,Lz,rx,ry):
    lam=0.075;xr=xc/rx;Lr=Lx/rx;ar=(Lx*Lz)/(rx*3)
    dx=xc-10.0;dy=ry+100.0;dz_=zc+10.0;d=np.sqrt(dx**2+dy**2+dz_**2);ax=dx/d;az=dz_/d
    mL=xc-Lx/2;mR=rx-(xc+Lx/2);mB=zc-Lz/2;mT=3-(zc+Lz/2);asp=Lx/(Lz+1e-6)
    px=(math.pi/lam)*Lx*ax;pz=(math.pi/lam)*Lz*az;LoL=Lx/lam;LzL=Lz/lam
    return np.column_stack([xc,zc,Lx,Lz,np.full_like(xc,rx),np.full_like(xc,ry),
        xr,Lr,ar,d,ax,az,mL,mR,mB,mT,asp,px,pz,LoL,LzL]).astype(np.float32)
def surr(X4,rx,ry):return lgbm_m.predict(enrich(X4[:,0],X4[:,1],X4[:,2],X4[:,3],rx,ry))

# ==========================================
# Physics Engine
# ==========================================
x_BS,y_BS,z_BS=10.0,-100.0,-10.0;E=8;d_B=0.075;lam=0.075;L1=2;d_ref=abs(y_BS)*1.5
P_BS_dBm=40.0;R_th=0.2;N0_dBm_Hz=-174.0;B=20e6;p_m=1./5.;n_users=5;room_z=3.0
P_BS=10**(P_BS_dBm/10.)*1e-3;N0=10**(N0_dBm_Hz/10.)*1e-3*B;Z_HS=[1.5,1.625,1.75,1.875,2.0];N_Z=5

def gen_rwp_traj(rx,ry,sim_time,dt=10,nu=5):
    rng=np.random;rng.seed(777)
    rs=[0.0,rx,0.0,ry]; hc=np.array([rx/2,ry/2]); hr=rx/3.0
    p_t=0.6;p_s=0.3;tau_h=1.5;tau_n=0.1;v_h=0.2;v_n=1.0;g_h=0.6;g_n=0.2
    ts=int(sim_time/dt)
    def gt(p):
        t=hc+(rng.rand(2)-0.5)*2*hr if rng.rand()<g_h else np.array([rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])])
        t[0]=np.clip(t[0],rs[0],rs[1]);t[1]=np.clip(t[1],rs[2],rs[3]);return t
    def ta(r):return tau_h if r=='hot' else tau_n
    def sp(r):return v_h if r=='hot' else v_n
    uh=1.5+0.5*rng.rand(nu);pos=np.zeros((nu,ts,2))
    up=np.zeros((nu,2));ur=[None]*nu;us=[None]*nu;ut_=np.zeros((nu,2));ud=np.zeros((nu,2));usp=np.zeros(nu);uprem=np.zeros(nu)
    for i in range(nu):
        up[i]=[rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])];d2h=np.linalg.norm(up[i]-hc);ur[i]='hot' if d2h<=hr else 'normal'
        if rng.rand()<p_t:us[i]='transfer';ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
        else:us[i]='pause';uprem[i]=rng.exponential(ta(ur[i]))
        pos[i,0,:]=up[i]
    for step in range(1,ts):
        for i in range(nu):
            if us[i]=='pause':
                uprem[i]-=dt;pos[i,step,:]=up[i]
                if uprem[i]<=0:us[i]='transfer';ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
            else:
                md=usp[i]*dt;pd=ut_[i]-up[i]
                if np.linalg.norm(pd)<=md:
                    up[i]=ut_[i].copy();d2h=np.linalg.norm(up[i]-hc);ur[i]='hot' if d2h<=hr else 'normal'
                    if rng.rand()<p_s:us[i]='pause';uprem[i]=rng.exponential(ta(ur[i]))
                    else:ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
                else:up[i]=up[i]+ud[i]*md
                pos[i,step,:]=up[i]
    pts=np.zeros((nu*ts,3));idx=0
    for u in range(nu):
        for s in range(ts):pts[idx]=[pos[u,s,0],pos[u,s,1],uh[u]];idx+=1
    return pts

_PC={}
def build_room(rx,ry):
    nx,ny=max(40,int(rx/0.05)),max(40,int(ry/0.05))
    xv=torch.linspace(0,rx,nx,device=device);yv=torch.linspace(0,ry,ny,device=device)
    Xg,Yg=torch.meshgrid(xv,yv,indexing='ij');xyf=torch.stack([Xg.flatten(),Yg.flatten()],dim=1)
    gl=[]
    for zh in Z_HS:gl.append(torch.stack([Xg.flatten(),Yg.flatten(),torch.full_like(Xg.flatten(),zh)],dim=1))
    gl=torch.cat(gl,dim=0);Ng=gl.shape[0]
    et=gen_rwp_traj(rx,ry,100000,10);kde=gaussian_kde(et[:,:2].T,bw_method='scott')
    wxy=kde(xyf.cpu().numpy().T).flatten();wxy=wxy/wxy.sum()
    w_full=np.tile(wxy,N_Z);gw=torch.tensor(w_full/w_full.sum(),dtype=torch.float32,device=device)
    np.random.seed(42)
    ps=torch.tensor(2*np.pi*np.random.rand(Ng),dtype=torch.float32,device=device)
    tt=torch.tensor(-np.pi+2*np.pi*np.random.rand(Ng),dtype=torch.float32,device=device)
    eta=torch.tensor(10**((-15+5*np.random.rand(Ng))/10),dtype=torch.float32,device=device)
    na=torch.arange(E,dtype=torch.float32,device=device);dyB=torch.tensor(0.0-y_BS,device=device)
    v1c=lam/(4*math.pi);v1pc=-(2*math.pi/lam);oE=1/math.sqrt(E)
    return gl,gw,ps,tt,eta,na,dyB,v1c,v1pc,oE

@torch.no_grad()
def phys_out(X4,rx,ry):
    k=(round(rx,2),round(ry,2))
    if k not in _PC:_PC[k]=build_room(rx,ry)
    gl,gw,ps,tt,eta,na,dyB,v1c,v1pc,oE=_PC[k]
    out=[]
    for i in range(0,len(X4),200):
        b=torch.tensor(X4[i:i+200],dtype=torch.float32,device=device);bo=torch.zeros(len(b),device=device)
        for j in range(len(b)):
            xc,zc,Lx,Lz=b[j,0],b[j,1],b[j,2],b[j,3];xg,yg,zg=gl[:,0],gl[:,1],gl[:,2];Ng=xg.shape[0]
            dx=xc-x_BS;dy=dyB;dz=zc-z_BS;Rw=torch.sqrt(dx**2+dy**2+dz**2);th=torch.atan2(dy,dx);ph=torch.acos(dz/Rw)
            kx=torch.sin(ph)*torch.cos(th);kz=torch.cos(ph)
            x1,z1=xc-Lx/2,zc-Lz/2;x2,z2=xc-Lx/2,zc+Lz/2;x3,z3=xc+Lx/2,zc-Lz/2;x4,z4=xc+Lx/2,zc+Lz/2
            def rd(xs,zs):dd=xs-x_BS;dz_=zs-z_BS;L=torch.sqrt(dd**2+dy**2+dz_**2);return dd/L,dy/L,dz_/L
            ux1,_,uz1=rd(x1,z1);ux2,_,uz2=rd(x2,z2);ux3,_,uz3=rd(x3,z3);ux4,_,uz4=rd(x4,z4)
            uma=torch.stack([ux1,ux2,ux3,ux4]);uzm=torch.stack([uz1,uz2,uz3,uz4])
            umin=uma.min(0).values;umax=uma.max(0).values;zmin=uzm.min(0).values;zmax=uzm.max(0).values
            dux=xg-x_BS;duy=yg-y_BS;duz=zg-z_BS;Lu=torch.sqrt(dux**2+duy**2+duz**2);uux=dux/Lu;uuz=duz/Lu
            ix=torch.sigmoid(1000*(uux-umin))*torch.sigmoid(1000*(umax-uux))
            iz=torch.sigmoid(1000*(uuz-zmin))*torch.sigmoid(1000*(zmax-uuz));il=ix*iz
            d2x=xg-xc;d2y=yg;d2z=zg-zc;Ru=torch.sqrt(d2x**2+d2y**2+d2z**2);t1=d2x/Ru;t2=d2z/Ru
            ax_=(1-il)*(kx-t1);az_=(1-il)*(kz-t2)
            sx=torch.sinc((math.pi/lam)*Lx*ax_);sz=torch.sinc((math.pi/lam)*Lz*az_)
            p1=(2*math.pi/lam)*d_B*na*torch.sin(th);a1=oE*torch.complex(torch.cos(p1),torch.sin(p1))
            v1m=v1c/Rw;v1p_=v1pc*Rw;v1=torch.complex(v1m*torch.cos(v1p_),v1m*torch.sin(v1p_));H1=v1*a1.conj()
            p2=(2*math.pi/lam)*d_B*na.unsqueeze(0)*torch.sin(tt).unsqueeze(1)
            a2=oE*torch.complex(torch.cos(p2),torch.sin(p2))
            v2m=eta*(lam/(4*math.pi*d_ref));v2=torch.complex(v2m*torch.cos(ps),v2m*torch.sin(ps))
            H2=v2.unsqueeze(1)*a2.conj();Ht=math.sqrt(E/L1)*(H1.unsqueeze(0)+H2)
            fm=(Lx*Lz*sx*sz)/(lam*Ru);fp=torch.tensor(-(2*math.pi/lam),device=device)*(kx*xc+kz*zc)-(math.pi/2)
            fc=torch.complex(fm*torch.cos(fp),fm*torch.sin(fp));He=Ht*fc.unsqueeze(1)
            Hw=torch.sum(He,dim=1)/math.sqrt(E);dp=(torch.abs(Hw)**2)*p_m*P_BS;it=(n_users-1)*dp;sr=dp/(it+N0)
            ab=math.pi/3.0;Ses=torch.zeros(Ng,device=device)
            rn=torch.sqrt((xg-xc)**2+yg**2+(zg-zc)**2+1e-12)
            for a in [0.0,math.pi/2,math.pi,3*math.pi/2]:
                dot=torch.abs((xg-xc)*math.cos(a)+yg*math.sin(a));cp=dot/rn;ph_b=torch.acos(torch.clamp(cp,0,1))
                Se=(math.pi-torch.clamp(2*ab-ph_b,min=0))/math.pi;Ses=Ses+torch.clamp(Se,1./3.,1.0)
            sr_se=((Ses/4)*dp)/((Ses/4)*it+N0);bits=(torch.log2(1+sr_se)<R_th).float();bo[j]=torch.sum(bits*gw)
        out.append(bo.cpu().numpy());torch.cuda.empty_cache()
    return np.concatenate(out)

# ==========================================
# Run 3-way validation on each room
# ==========================================
ROOMS=[(8,6),(12,10),(15,12)]
N_GEN=200
results=[]

for rx,ry in ROOMS:
    print(f'\n{"="*60}')
    print(f'Room: {rx}x{ry}m')
    print('='*60)
    lb=np.array([0.,0.,.05,.05]);ub=np.array([rx,room_z,rx,room_z])
    row={'room':f'{rx}x{ry}'}

    # --- A: Pure NSGA-II (ground truth) ---
    class P_Phys(ElementwiseProblem):
        def __init__(self):super().__init__(n_var=4,n_obj=2,n_ieq_constr=4,xl=lb,xu=ub)
        def _evaluate(self,x,out,*a,**k):
            xc,zc,Lx,Lz=x;out["G"]=[Lx/2-xc,xc+Lx/2-rx,Lz/2-zc,zc+Lz/2-room_z]
            out["F"]=[Lx*Lz,float(phys_out(x.reshape(1,-1),rx,ry)[0])]
    print('A: Pure NSGA-II...')
    t0=time.time()
    algo_a=NSGA2(pop_size=300,n_offsprings=150,sampling=FloatRandomSampling(),crossover=SBX(.9,15),mutation=PM(.25,20),eliminate_duplicates=True)
    res_a=minimize(P_Phys(),algo_a,get_termination('n_gen',N_GEN),seed=42,verbose=False)
    row['t_phys']=time.time()-t0;Fa=res_a.F
    row['knee_phys']=f'{Fa[Fa[:,1]<=0.10,0].min():.4f} m²' if (Fa[:,1]<=0.10).any() else 'N/A'
    print(f'  {row["t_phys"]:.0f}s, {len(Fa)} pts, Knee={row["knee_phys"]}')

    # --- B: LGBM+NSGA-II surrogate, physics-verified ---
    class P_Surr(ElementwiseProblem):
        def __init__(self):super().__init__(n_var=4,n_obj=2,n_ieq_constr=4,xl=lb,xu=ub)
        def _evaluate(self,x,out,*a,**k):
            xc,zc,Lx,Lz=x;out["G"]=[Lx/2-xc,xc+Lx/2-rx,Lz/2-zc,zc+Lz/2-room_z]
            out["F"]=[Lx*Lz,float(surr(x.reshape(1,-1),rx,ry)[0])]
    print('B: LGBM+NSGA-II...')
    t0=time.time()
    algo_b=NSGA2(pop_size=300,n_offsprings=150,sampling=FloatRandomSampling(),crossover=SBX(.9,15),mutation=PM(.25,20),eliminate_duplicates=True)
    res_sn=minimize(P_Surr(),algo_b,get_termination('n_gen',N_GEN),seed=42,verbose=False)
    Fb=np.column_stack([res_sn.X[:,2]*res_sn.X[:,3],phys_out(res_sn.X,rx,ry)])
    row['t_nsga2']=time.time()-t0;row['knee_nsga2']=f'{Fb[Fb[:,1]<=0.10,0].min():.4f} m²' if (Fb[:,1]<=0.10).any() else 'N/A'
    print(f'  {row["t_nsga2"]:.0f}s, Knee={row["knee_nsga2"]}')

    # --- C: LGBM+AGE-MOEA surrogate, physics-verified ---
    print('C: LGBM+AGE-MOEA...')
    t0=time.time()
    algo_c=AGEMOEA(pop_size=300,n_offsprings=150,sampling=FloatRandomSampling(),crossover=SBX(.9,15),mutation=PM(.25,20))
    res_sa=minimize(P_Surr(),algo_c,get_termination('n_gen',N_GEN),seed=42,verbose=False)
    Fc=np.column_stack([res_sa.X[:,2]*res_sa.X[:,3],phys_out(res_sa.X,rx,ry)])
    row['t_age']=time.time()-t0;row['knee_age']=f'{Fc[Fc[:,1]<=0.10,0].min():.4f} m²' if (Fc[:,1]<=0.10).any() else 'N/A'
    print(f'  {row["t_age"]:.0f}s, Knee={row["knee_age"]}')

    # HV/IGD
    nd=np.max(np.vstack([Fa,Fb,Fc]),axis=0)*1.1;id_=np.min(np.vstack([Fa,Fb,Fc]),axis=0)*.9
    Fan=(Fa-id_)/(nd-id_);Fbn=(Fb-id_)/(nd-id_);Fcn=(Fc-id_)/(nd-id_);rp=np.array([1.1,1.1])
    hv_a=HV(ref_point=rp)(Fan)
    row['hv_nsga2']=abs(hv_a-HV(ref_point=rp)(Fbn))/hv_a*100
    row['hv_age']=abs(hv_a-HV(ref_point=rp)(Fcn))/hv_a*100
    row['igd_nsga2']=IGD(Fan)(Fbn);row['igd_age']=IGD(Fan)(Fcn)
    results.append(row)

    # Plot
    fig,ax=plt.subplots(figsize=(9,6))
    ax.scatter(Fa[:,1]*100,Fa[:,0],c='navy',s=8,alpha=.5,label=f'Pure NSGA-II ({len(Fa)}pts)')
    ax.scatter(Fb[:,1]*100,Fb[:,0],c='green',s=8,alpha=.5,marker='s',label=f'LGBM+NSGA2 ({len(Fb)}pts)')
    ax.scatter(Fc[:,1]*100,Fc[:,0],c='orange',s=8,alpha=.5,marker='^',label=f'LGBM+AGE ({len(Fc)}pts)')
    ax.axvline(x=10,color='red',ls='--',alpha=.5)
    ax.set_xlabel('Outage[%]');ax.set_ylabel('Area[m²]')
    ax.set_title(f'{rx}x{ry}m | HV: NSGA2={row["hv_nsga2"]:.1f}% AGE={row["hv_age"]:.1f}%')
    ax.legend(fontsize=8);ax.grid(alpha=.3)
    plt.tight_layout();plt.savefig(f'val_3way_{rx}x{ry}.png',dpi=120);plt.show()

# Summary
print(f'\n{"Room":<10} {"Pure t":<8} {"Knee":<12} {"NSGA2 t":<8} {"HV":<8} {"IGD":<8} {"AGE t":<8} {"HV":<8} {"IGD":<8}')
print('-'*85)
for r in results:
    print(f'{r["room"]:<10} {r["t_phys"]:<8.0f}s {r["knee_phys"]:<12} {r["t_nsga2"]:<8.0f}s {r["hv_nsga2"]:<8.1f}% {r["igd_nsga2"]:<8.4f} {r["t_age"]:<8.0f}s {r["hv_age"]:<8.1f}% {r["igd_age"]:<8.4f}')

from IPython.display import FileLink,display
for rx,ry in ROOMS:display(FileLink(f'val_3way_{rx}x{ry}.png'))